# EDA for a low-rate power series, with a domain-shift judge

Companion to `eda_shelly.py` (same battery, same verdict logic -- this notebook is the interactive twin).

**Why.** When a real Shelly CSV arrives, the question is never "is it pretty" but "does it behave like a real house at the rate this sensor delivers". This notebook answers that with a fixed battery of statistics, judged against a reference computed on real data (UK-DALE slice: 6 s, one UK home, 70.66 days) and stored as `eda_reference_ukdale.json`.

**What it produces** (in `OUTDIR`): `report.md`, `metrics.json`, four figures that are the working analogues of Figures 1-4 in `docs/research/nilm-visual-reading.md` (resolution ladder, the week, one day + appliances, event view).

**Typical sessions**
1. First contact with any new CSV: set `INPUT` to the path, `REFERENCE_PATH` to `eda_reference_ukdale.json` -- read the verdict table at the bottom.
2. No appliances labeled / want only the whole-house view: same as above; the per-appliance tables come out empty and that is fine.
3. Rebuilding the reference (only for the UK-DALE slice): `INPUT = 'ukdale'`, `MAKE_REFERENCE = True`.


## 0 -- Configure

Edit only this cell. Everything below runs top to bottom.


In [ ]:
# INPUT: 'ukdale' loads the local UK-DALE slice; anything else is treated as a CSV path.
# CSV expectations: a timestamp column and a whole-house power column (active_power_W / power_W / power / ...),
# plus optional per-appliance columns named <name>_power_W. The *_on flags are ignored (ON states are
# recomputed from power), so the notebook is rate-agnostic and label-free.
INPUT = 'ukdale'
OUTDIR = 'eda_out_notebook'
REFERENCE_PATH = None        # e.g. 'eda_reference_ukdale.json' to judge against the UK-DALE reference
MAKE_REFERENCE = False       # True only when INPUT='ukdale' and you want to (re)build the reference


In [ ]:
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(OUTDIR, exist_ok=True)
pd.set_option('display.width', 140)

# display() exists in real Jupyter; fall back to print when executed as plain Python
try:
    display
except NameError:
    def display(x):
        print(x)


## 1 -- Load the input

Auto-detection: the power column is matched from a known list, per-appliance columns are anything matching `<name>_power_W`.

For `ukdale`: channels 2-6 are the labeled appliances (per `labels.dat`); channel 1 is the household's own `custom_aggregate` -- **not** the house mains. Reconciliation on this slice: aggregate 306.0 kWh vs 261.8 kWh across the labeled channels, i.e. **14.5% of the aggregate is unlabeled energy** -- the UNKNOWN bucket is real even here. The full UK-DALE house-1 download adds the true mains and the proper residual.


In [ ]:
POWER_KEYS = ['active_power_W', 'power_W', 'Power_W', 'power', 'active_power', 'load_W']
TS_KEYS = ['timestamp', 'Timestamp', 'time', 'Time', 'datetime']
APP_RE = re.compile(r'^([A-Za-z0-9_]+)_power_?[Ww]$')

UKDALE_CANDIDATES = [
    'projects/watt-wiser/research-logs/sakunrasilka_nilm-test2',
    '/root/longvdh/agent_space/projects/watt-wiser/research-logs/sakunrasilka_nilm-test2',
]
UKDALE_CH = {2: 'fridge', 3: 'dish_washer', 4: 'kettle', 5: 'washing_machine', 6: 'monitor'}

def load_ukdale():
    base = next((b for b in UKDALE_CANDIDATES if os.path.isdir(b)), None)
    if base is None:
        raise FileNotFoundError('UK-DALE slice not found in: ' + ', '.join(UKDALE_CANDIDATES))
    arrays = {c: np.loadtxt(os.path.join(base, 'channel_%d.dat' % c), ndmin=2) for c in range(1, 7)}
    cstart = max(a[0, 0] for a in arrays.values())
    cend = min(a[-1, 0] for a in arrays.values())
    keep = {c: a[(a[:, 0] >= cstart) & (a[:, 0] <= cend)] for c, a in arrays.items()}
    n = min(len(v) for v in keep.values())
    df = pd.DataFrame({'ts': pd.to_datetime(keep[1][:n, 0], unit='s'),
                       'power': keep[1][:n, 1].astype(float)})
    apps = {nm: keep[c][:n, 1].astype(float) for c, nm in UKDALE_CH.items()}
    return df, apps

def load_csv(path):
    raw = pd.read_csv(path)
    ts_col = next((c for c in TS_KEYS if c in raw.columns), raw.columns[0])
    pow_col = next((c for c in POWER_KEYS if c in raw.columns), None)
    if pow_col is None:
        cand = [c for c in raw.columns if 'power' in c.lower() and not c.endswith('_on')]
        if not cand:
            raise ValueError('no power column found; columns: ' + str(list(raw.columns)))
        pow_col = cand[0]
    app_cols = {}
    for c in raw.columns:
        mth = APP_RE.match(c)
        if mth and c != pow_col:
            app_cols[mth.group(1)] = raw[c].values.astype(float)
    return pd.DataFrame({'ts': pd.to_datetime(raw[ts_col]), 'power': raw[pow_col].values.astype(float)}), app_cols

if INPUT == 'ukdale':
    df, apps = load_ukdale()
else:
    df, apps = load_csv(INPUT)

print(len(df), 'rows |', df['ts'].min(), '->', df['ts'].max())
print('appliance columns:', list(apps) or 'none (whole-house only)')


## 2 -- Primitives

- `run_lengths`: True-run starts and lengths (for duty and ON-duration statistics).
- `on_mask`: ON = power above a per-appliance floor (5 W or 2% of p95, whichever is larger) -- rate-agnostic, label-free.
- `psi`: Population Stability Index over quantile bins of the reference (<0.1 none, 0.1-0.25 moderate, >0.25 major; the credit-scoring convention).


In [ ]:
def run_lengths(mask):
    d = np.diff(mask.astype(np.int8))
    starts = np.where(d == 1)[0] + 1
    ends = np.where(d == -1)[0] + 1
    if mask[0]:
        starts = np.r_[0, starts]
    if mask[-1]:
        ends = np.r_[ends, len(mask)]
    if len(starts) == 0:
        return np.empty(0, int), np.empty(0, int)
    return starts, ends - starts

def psi(new, ref, bins=10):
    ref = np.asarray(ref, float); new = np.asarray(new, float)
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0] = -np.inf; edges[-1] = np.inf
    edges = np.maximum.accumulate(edges)
    refp = np.histogram(ref, edges)[0] / max(len(ref), 1)
    newp = np.histogram(new, edges)[0] / max(len(new), 1)
    refp = np.clip(refp, 1e-6, None); newp = np.clip(newp, 1e-6, None)
    return float(np.sum((newp - refp) * np.log(newp / refp)))

def on_mask(ap):
    p95 = np.percentile(ap, 95) if len(ap) else 0.0
    return ap > max(5.0, 0.02 * p95)


## 3 -- The battery

Fixed metrics, no tuning knobs: sampling (dt, gaps), power percentiles + always-on estimate (median of daily 10th percentiles), step texture (steps/day above 30/100/300 W, quiet-regime noise floor), diurnal structure (hourly profile, peak/trough, lag-1-day autocorrelation, weekday vs weekend), per-appliance duty/energy/events/durations/transition magnitudes, and simultaneity (share of time with 2+ appliances ON -- the separation-difficulty axis).


In [ ]:
def compute_metrics(df, apps):
    m = {}
    ts = df['ts'].values
    p = df['power'].values.astype(float)
    n = len(p)
    sec = (ts[-1] - ts[0]) / np.timedelta64(1, 's')
    days = sec / 86400.0
    dt_all = np.diff(ts) / np.timedelta64(1, 's')
    dt_med = float(np.median(dt_all))
    m['sampling'] = {
        'rows': int(n), 'span_days': round(days, 2), 'dt_median_s': dt_med,
        'dt_min_s': float(dt_all.min()), 'dt_max_s': float(dt_all.max()),
        'gap_fraction': round(float(np.mean(dt_all > 3 * dt_med)), 4),
    }
    pct = np.percentile(p, [1, 5, 25, 50, 75, 95, 99])
    dp10 = df.groupby(df['ts'].dt.date)['power'].quantile(0.10).dropna()
    m['power'] = {
        'mean_W': round(float(p.mean()), 1), 'p1_W': round(float(pct[0]), 1),
        'p5_W': round(float(pct[1]), 1), 'p25_W': round(float(pct[2]), 1),
        'p50_W': round(float(pct[3]), 1), 'p75_W': round(float(pct[4]), 1),
        'p95_W': round(float(pct[5]), 1), 'p99_W': round(float(pct[6]), 1),
        'max_W': round(float(p.max()), 1),
        'always_on_W': round(float(np.median(dp10.values)), 1) if len(dp10) else round(float(pct[1]), 1),
    }
    d = np.abs(np.diff(p))
    quiet = (p[:-1] < pct[2]) & (p[1:] < pct[2])
    m['steps'] = {
        'dP_p50_W': round(float(np.percentile(d, 50)), 1),
        'dP_p90_W': round(float(np.percentile(d, 90)), 1),
        'dP_p99_W': round(float(np.percentile(d, 99)), 1),
        'noise_floor_W': round(float(np.median(d[quiet])), 1) if quiet.any() else None,
        'steps_per_day_gt30W': round(float(np.sum(d > 30) / days), 1),
        'steps_per_day_gt100W': round(float(np.sum(d > 100) / days), 1),
        'steps_per_day_gt300W': round(float(np.sum(d > 300) / days), 1),
    }
    hr = df['ts'].dt.hour.values
    prof = [float(p[hr == h].mean()) for h in range(24)]
    L = int(round(86400.0 / dt_med))
    m['diurnal'] = {
        'hourly_mean_W': [round(x, 1) for x in prof],
        'peak_hour': int(np.argmax(prof)), 'trough_hour': int(np.argmin(prof)),
        'peak_over_trough': round(float(max(prof) / max(min(prof), 1e-9)), 2),
    }
    if n > 2 * L:
        m['diurnal']['lag1day_autocorr'] = round(float(np.corrcoef(p[:-L], p[L:])[0, 1]), 3)
    wd = df['ts'].dt.dayofweek.values
    m['diurnal']['weekday_mean_W'] = round(float(p[wd < 5].mean()), 1)
    m['diurnal']['weekend_mean_W'] = round(float(p[wd >= 5].mean()), 1)
    tot_e = float(p.sum() * dt_med)
    m['appliances'] = {}
    masks = {}
    for nm, ap in apps.items():
        ap = np.asarray(ap, float)
        mask = on_mask(ap)
        masks[nm] = mask
        starts, lens = run_lengths(mask)
        on_p = ap[mask]
        durs = lens * dt_med / 60.0
        dap = np.abs(np.diff(ap))
        tr = np.where(np.diff(mask.astype(np.int8)) != 0)[0]
        m['appliances'][nm] = {
            'energy_share_pct': round(100 * float(ap.sum() * dt_med) / tot_e, 1) if tot_e else None,
            'duty_pct': round(100 * float(mask.mean()), 1),
            'median_on_W': round(float(np.median(on_p)), 0) if len(on_p) else 0,
            'events_per_day': round(float(len(lens) / days), 2) if days else 0,
            'on_minutes_p50': round(float(np.percentile(durs, 50)), 1) if len(durs) else None,
            'on_minutes_p90': round(float(np.percentile(durs, 90)), 1) if len(durs) else None,
            'transition_step_p50_W': round(float(np.median(dap[tr])), 0) if len(tr) else None,
        }
    if masks:
        non = np.zeros(n, int)
        for mask in masks.values():
            non += mask.astype(int)
        m['overlap'] = {
            'frac_0_on_pct': round(100 * float(np.mean(non == 0)), 1),
            'frac_1_on_pct': round(100 * float(np.mean(non == 1)), 1),
            'frac_2plus_on_pct': round(100 * float(np.mean(non >= 2)), 1),
        }
    return m

m = compute_metrics(df, apps)


## 4 -- Results


In [ ]:
for key in ['sampling', 'power', 'steps', 'diurnal']:
    d = dict(m[key]); d.pop('hourly_mean_W', None)
    print('---', key)
    display(pd.DataFrame([d]).T.rename(columns={0: 'value'}))
if m['appliances']:
    print('--- appliances')
    display(pd.DataFrame(m['appliances']).T)
if 'overlap' in m:
    print('--- simultaneity')
    display(pd.DataFrame([m['overlap']]))


## 5 -- The four figures (analogues of Figures 1-4)

fig01: one day at three sampling rates (the ladder) -- fig02: the week unrolled -- fig03: one day with appliances overlaid -- fig04: the event view (step sizes, hourly profile, per-appliance transition ECDF, ON-duration ECDF).


In [ ]:
def make_figures(df, apps):
    p = df['power'].values.astype(float)
    ts = df['ts']
    day0 = ts.min().normalize()
    d0 = df[(ts >= day0) & (ts < day0 + pd.Timedelta(days=1))]
    dt_med = float(np.median(np.diff(ts.values) / np.timedelta64(1, 's')))
    # fig01: resolution ladder
    fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    y0 = d0['power'].values
    for ax, (ttl, k) in zip(axes, [('native', 1), ('60 s', max(int(round(60 / dt_med)), 1)),
                                   ('300 s', max(int(round(300 / dt_med)), 1))]):
        if k <= 1:
            ax.plot(d0['ts'], y0, lw=0.6, color='tab:red')
        else:
            kk = min(k, max(len(y0) // 2, 1))
            yb = y0[:len(y0) // kk * kk].reshape(-1, kk).mean(axis=1)
            ax.plot(d0['ts'].values[::kk][:len(yb)], yb, lw=0.8, color='tab:red')
        ax.set_ylabel('W'); ax.set_title('sampling: %s' % ttl, fontsize=9)
    fig.suptitle('fig01 analogue: one day at three sampling rates (the ladder)')
    fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'eda_fig01_ladder.png'), dpi=150); plt.show(); plt.close(fig)
    # fig02: the week
    wk = df[ts < ts.min() + pd.Timedelta(days=7)]
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(wk['ts'], wk['power'], lw=0.4, color='tab:blue')
    ax.set_ylabel('W'); ax.set_title('fig02 analogue: the week unrolled')
    fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'eda_fig02_week.png'), dpi=150); plt.show(); plt.close(fig)
    # fig03: one day with appliances
    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.plot(d0['ts'], d0['power'], lw=0.5, color='k', label='whole house')
    for nm, ap in apps.items():
        ax.plot(d0['ts'].values, np.asarray(ap, float)[:len(d0)], lw=0.4, alpha=0.7, label=nm)
    ax.legend(fontsize=7, ncol=6); ax.set_ylabel('W')
    ax.set_title('fig03 analogue: one day, whole house + appliances')
    fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'eda_fig03_day.png'), dpi=150); plt.show(); plt.close(fig)
    # fig04: event view
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    ax = axes[0][0]
    dd = np.abs(np.diff(p)); dd = dd[dd > 0.5]
    if len(dd):
        ax.hist(dd, bins=np.logspace(0, np.log10(max(dd.max(), 10.0)), 60), color='tab:red')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title('(a) whole-house step sizes |dP|'); ax.set_xlabel('W')
    ax = axes[0][1]
    prof = [float(p[df['ts'].dt.hour.values == h].mean()) for h in range(24)]
    ax.plot(range(24), prof, 'o-', color='tab:blue', ms=3)
    ax.set_title('(b) hourly profile'); ax.set_xlabel('hour'); ax.set_ylabel('mean W')
    ax = axes[1][0]
    for nm, ap in apps.items():
        ap = np.asarray(ap, float); mask = on_mask(ap)
        tr = np.where(np.diff(mask.astype(np.int8)) != 0)[0]
        if len(tr):
            xs = np.sort(np.abs(np.diff(ap))[tr])
            ax.plot(xs, np.linspace(0, 1, len(xs)), label=nm, lw=1.2)
    ax.set_xscale('log'); ax.set_title('(c) appliance transition steps, ECDF'); ax.set_xlabel('|dP| W'); ax.legend(fontsize=7)
    ax = axes[1][1]
    for nm, ap in apps.items():
        ap = np.asarray(ap, float); mask = on_mask(ap)
        starts, lens = run_lengths(mask)
        if len(lens):
            xs = np.sort(lens) * dt_med / 60.0
            ax.plot(xs, np.linspace(0, 1, len(xs)), label=nm, lw=1.2)
    ax.set_xscale('log'); ax.set_title('(d) ON durations, ECDF'); ax.set_xlabel('minutes'); ax.legend(fontsize=7)
    fig.suptitle('fig04 analogue: reading events, not shapes')
    fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'eda_fig04_events.png'), dpi=150); plt.show(); plt.close(fig)

make_figures(df, apps)


## 6 -- Domain-shift judge

Each metric is compared against the UK-DALE reference with explicit ratio bands; the 24-hour profile additionally gets a PSI. **PASS** = within the band (the data behaves like the reference on that axis); **FLAG** = it does not. The reference was built from the UK-DALE slice (6 s, 70.66 days) and reproduces the hand-established numbers (715 steps/day > 30 W, 32.7% two-plus-ON, monitor duty 82.1%, kettle 2,890 W).


In [ ]:
def compare(m, ref):
    rows = []
    def row(metric, a, b, lo, hi, fmt='{:.2f}'):
        if a is None or b is None:
            return
        ratio = float(a) / float(b) if b else float('inf')
        verdict = 'PASS' if lo <= ratio <= hi else 'FLAG'
        rows.append({'metric': metric, 'input': fmt.format(a), 'reference': fmt.format(b),
                     'ratio': round(ratio, 2), 'verdict': verdict})
    s, rs = m['sampling'], ref['sampling']
    row('dt median (s)', s['dt_median_s'], rs['dt_median_s'], 0.99, 1.01, '{:.1f}')
    row('span (days)', s['span_days'], rs['span_days'], 0.3, 30.0, '{:.1f}')
    pw, rpw = m['power'], ref['power']
    row('mean power (W)', pw['mean_W'], rpw['mean_W'], 0.3, 3.0, '{:.0f}')
    row('always-on estimate (W)', pw['always_on_W'], rpw['always_on_W'], 0.25, 4.0, '{:.0f}')
    st, rst = m['steps'], ref['steps']
    row('steps/day >30 W', st['steps_per_day_gt30W'], rst['steps_per_day_gt30W'], 0.25, 4.0, '{:.1f}')
    row('steps/day >300 W', st['steps_per_day_gt300W'], rst['steps_per_day_gt300W'], 0.1, 10.0, '{:.1f}')
    row('noise floor (W)', st['noise_floor_W'], rst['noise_floor_W'], 0.25, 4.0, '{:.1f}')
    row('lag-1day autocorr', m['diurnal'].get('lag1day_autocorr'),
        ref['diurnal'].get('lag1day_autocorr'), 0.05, 20.0, '{:.2f}')
    if 'overlap' in m and 'overlap' in ref:
        row('2+ appliances ON (%)', m['overlap']['frac_2plus_on_pct'],
            ref['overlap']['frac_2plus_on_pct'], 0.25, 4.0, '{:.1f}')
    pv = psi(np.array(m['diurnal']['hourly_mean_W']), np.array(ref['diurnal']['hourly_mean_W']), bins=8)
    rows.append({'metric': 'diurnal profile PSI', 'input': '%.3f' % pv, 'reference': '0.000', 'ratio': None,
                 'verdict': 'PASS' if pv < 0.1 else ('MODERATE' if pv < 0.25 else 'FLAG')})
    return rows

cmp_rows = None
if REFERENCE_PATH:
    ref = json.load(open(REFERENCE_PATH))
    cmp_rows = compare(m, ref)
    display(pd.DataFrame(cmp_rows))
    flags = [r['metric'] for r in cmp_rows if r['verdict'] == 'FLAG']
    print('FLAG (%d of %d): %s' % (len(flags), len(cmp_rows), '; '.join(flags) or 'none'))
else:
    print('no REFERENCE_PATH -- skipping the judge (set it to eda_reference_ukdale.json to enable)')


## 7 -- Persist: report.md, metrics.json, reference


In [ ]:
def fmt_table(rows, header):
    out = ['| ' + ' | '.join(header) + ' |', '|' + '|'.join(['---'] * len(header)) + '|']
    for r in rows:
        out.append('| ' + ' | '.join(str(r.get(h, '')) for h in header) + ' |')
    return '\n'.join(out)

def write_report(m, cmp_rows, outdir, src, ref_name):
    L = ['# EDA report -- ' + src]
    if ref_name:
        L += ['', 'Compared against reference: %s (UK-DALE slice, 6 s, one UK home, 70.66 days).' % ref_name]
    L += ['', 'Generated by eda_shelly.ipynb (projects/watt-wiser/). Fixed metrics battery; PASS/FLAG are ratio bands '
          'around the UK-DALE reference; PSI follows the credit-scoring convention (<0.1 none, 0.1-0.25 moderate, >0.25 major).', '']
    L += ['## Sampling', '']
    L += [fmt_table([m['sampling']], ['rows', 'span_days', 'dt_median_s', 'dt_min_s', 'dt_max_s', 'gap_fraction']), '']
    L += ['## Power', '']
    L += [fmt_table([m['power']], list(m['power'].keys())), '']
    L += ['## Steps and events', '']
    L += [fmt_table([m['steps']], list(m['steps'].keys())), '']
    dd = {k: v for k, v in m['diurnal'].items() if k != 'hourly_mean_W'}
    L += ['## Diurnal', '']
    L += [fmt_table([dd], list(dd.keys())), '']
    if m['appliances']:
        L += ['## Appliances', '']
        first = next(iter(m['appliances'].values()))
        L += [fmt_table([dict(r, appliance=nm) for nm, r in m['appliances'].items()],
                        ['appliance'] + list(first.keys())), '']
    if 'overlap' in m:
        L += ['## Simultaneity', '']
        L += [fmt_table([m['overlap']], list(m['overlap'].keys())), '']
    if cmp_rows is not None:
        n_flag = sum(1 for r in cmp_rows if r['verdict'] == 'FLAG')
        L += ['## Domain shift vs reference', '']
        L += ['%d of %d metrics FLAG. PASS = within the ratio band of the UK-DALE reference; '
              'FLAG = this data does not behave like the reference on that axis.' % (n_flag, len(cmp_rows)), '']
        L += [fmt_table(cmp_rows, ['metric', 'input', 'reference', 'ratio', 'verdict']), '']
    open(os.path.join(outdir, 'report.md'), 'w').write('\n'.join(L))

write_report(m, cmp_rows, OUTDIR, INPUT, REFERENCE_PATH)
json.dump(m, open(os.path.join(OUTDIR, 'metrics.json'), 'w'), indent=1)
if MAKE_REFERENCE:
    json.dump(m, open(os.path.join(OUTDIR, 'reference_metrics.json'), 'w'), indent=1)
    print('reference written to %s/reference_metrics.json -- copy over eda_reference_ukdale.json when happy' % OUTDIR)
print('report + metrics written to', OUTDIR)


## 8 -- How to read the verdicts when real data arrives

- **dt FLAG is expected** on a 300 s Shelly feed -- it is the sampling-rate question answered in numbers, not a defect. Pair it with the fig01 ladder to see what 300 s does to a kettle event (one sample) vs the fridge cycle (about 4 samples).
- **noise floor** tells you which regime the house is in: UK-DALE quiet mains is 1.0 W at 6 s; the synthetic set random-walks at 14.6 W. A real Shelly between the two is normal; far above 15 W means installation noise (loose clamp, shared circuit).
- **2+ appliances ON** is the separation-difficulty fingerprint: UK-DALE 32.7%, synthetic 1.7%. A real home will land near the real side; if it does not, the labeled-inventory assumption needs re-checking before any model work.
- **PSI** on the hourly profile is the daily-shape fingerprint; expect a real Vietnamese home to MODERATE/FLAG against the UK reference on *shape* (a market difference, not a data-quality problem) while PASSing on *texture* (steps, noise, overlap).
- Event detection here is deliberately simple (thresholds + run-length encoding); the production upgrade path is PELT via `ruptures` -- see `analog-problems.md` section 2.5. EDA and domain-shift judging do not need it.

Full strategy context: `experiment-data-strategy.md` (same folder). CLI twin: `eda_shelly.py`.
